In [51]:
import os
import re
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F

# ── 1. LOAD & CLEAN ──────────────────────────────────────────
text = ""
for filename in os.listdir("books"):
    with open(f"books/{filename}", "r", encoding="utf-8", errors="ignore") as f:
        text += f.read()
text = re.sub(r'[^\x00-\x7F]', '', text)
print(f"Total characters: {len(text):,}")

# ── 2. VOCABULARY ────────────────────────────────────────────
chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}
print(f"Vocabulary size: {vocab_size}")

# ── 3. ENCODE & TRAINING PAIRS ───────────────────────────────
sequence_length = 10
encoded = [char_to_idx[c] for c in text]
X, y = [], []
for i in range(len(encoded) - sequence_length):
    X.append(encoded[i:i+sequence_length])
    y.append(encoded[i+sequence_length])
X = torch.tensor(X)
y = torch.tensor(y)
print(f"Training pairs: {len(X):,}")

# ── 4. MODEL ─────────────────────────────────────────────────
class CharModel(nn.Module):
    def __init__(self):
        super(CharModel, self).__init__()
        self.embed = nn.Embedding(vocab_size, 128)
        self.fc1 = nn.Linear(sequence_length * 128, 512)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(512, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = CharModel().cuda()
print(model)

# ── 5. TRAIN ─────────────────────────────────────────────────
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=256, shuffle=True)
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()

for epoch in range(5):
    for i, (x_batch, y_batch) in enumerate(dataloader):
        x_batch = x_batch.cuda()
        y_batch = y_batch.cuda()
        y_pred = model(x_batch)
        loss = loss_fn(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if i % 1000 == 0:
            print(f"Epoch {epoch+1} | batch {i}/{len(dataloader)} | loss {loss.item():.4f}")
    print(f"--- Epoch {epoch+1} done ---")

# ── 6. GENERATE ──────────────────────────────────────────────
def generate(prompt, num_chars=200, temperature=0.8):
    model.eval()
    input_chars = [char_to_idx.get(c, 0) for c in prompt]
    while len(input_chars) < sequence_length:
        input_chars = [0] + input_chars
    input_chars = input_chars[-sequence_length:]
    result = prompt
    for _ in range(num_chars):
        x = torch.tensor([input_chars]).cuda()
        with torch.no_grad():
            output = model(x)
        probs = F.softmax(output / temperature, dim=1)
        next_idx = torch.multinomial(probs, 1).item()
        result += idx_to_char[next_idx]
        input_chars = input_chars[1:] + [next_idx]
    print(result)

Total characters: 9,518,708
Vocabulary size: 127
Training pairs: 9,518,698
CharModel(
  (embed): Embedding(127, 128)
  (fc1): Linear(in_features=1280, out_features=512, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=512, out_features=127, bias=True)
)
Epoch 1 | batch 0/37183 | loss 4.8799
Epoch 1 | batch 1000/37183 | loss 2.5080
Epoch 1 | batch 2000/37183 | loss 2.3557
Epoch 1 | batch 3000/37183 | loss 2.1575
Epoch 1 | batch 4000/37183 | loss 1.9788
Epoch 1 | batch 5000/37183 | loss 2.2097
Epoch 1 | batch 6000/37183 | loss 2.0187
Epoch 1 | batch 7000/37183 | loss 2.0445
Epoch 1 | batch 8000/37183 | loss 2.0030
Epoch 1 | batch 9000/37183 | loss 2.0525
Epoch 1 | batch 10000/37183 | loss 2.0444
Epoch 1 | batch 11000/37183 | loss 2.3146
Epoch 1 | batch 12000/37183 | loss 2.2386
Epoch 1 | batch 13000/37183 | loss 2.1130
Epoch 1 | batch 14000/37183 | loss 2.1100
Epoch 1 | batch 15000/37183 | loss 2.1834
Epoch 1 | batch 16000/37183 | loss 2.1774
Epoch 1 | batch 17000/37183 | loss 2.1

In [55]:
generate("The ship")

The shipcs of unitzer, and in the same him were written stimple, indeed and though he was all that of the old
leaving the trates to get articley toward

laters and place should have spreadible in other given 
